# Pretraining on Modal, triggered from a Kaggle notebook

Train the base model on a Modal T4 card, keep the run directory on a Modal
Volume, and publish the `base` stage to the Hugging Face Hub. This notebook is
the client. The training binary runs in a Modal container.

The image bakes the repository and the shared library at image build time, so
the run pays no build cost. The container runs one T4 card (`sm_75`), which is
the same architecture as the Kaggle session.

The shape is the same 252M-parameter model as `pretrain.ipynb`. One card holds
the whole batch, so the batch is accumulated on one replica instead of two.
A full run takes longer than the two-card run and costs Modal container time.

Before the first run, deploy the app once from a workstation:

```bash
pip install modal && modal setup
modal deploy tools/modal/app.py
modal run tools/modal/app.py::doctor
```

`doctor` must report a T4 card and `sm_75`. See `docs/host-portability.md`.

## Setup

The setup has four parts. It installs the Modal client, reads three Kaggle
secrets, points the session at the `modal` run host, and clones and builds the
repository. It also checks the two account-level prerequisites.

Add three secrets to the notebook. `HF_TOKEN` comes from Hugging Face.
`MODAL_TOKEN_ID` and `MODAL_TOKEN_SECRET` come from the tokens page of the
Modal dashboard. The Modal client reads the two token values from the
environment, so no interactive login is needed.

Two actions are not part of the notebook, because they outlive one session:

| Action | When | Command |
|---|---|---|
| Deploy the app | Once per commit | `modal deploy tools/modal/app.py` |
| Create the Hub Secret | Once per account | `modal secret create huggingface HF_TOKEN=hf_...` |

The deploy carries the code that trains, so it must follow a commit. The
preflight stops with the command when either is absent.

In [ ]:
# The repository must exist before this module imports. Clone it, add the
# package paths, then let the session module own the host setup and the build.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ming6ao/nanochat.cpp"
REPO = Path("/kaggle/working/nanochat.cpp")
BASE = Path("/kaggle/working/nanochat-cache")
RUNS = Path("/kaggle/working/runs")

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
sys.path[:0] = [str(REPO / "python"), str(REPO / "notebooks")]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "modal"],
               check=True)

import session

session.load_secrets(session.SECRET_NAMES + ("MODAL_TOKEN_ID",
                                            "MODAL_TOKEN_SECRET"))

# One switch selects the run host for this session. The session reads it to
# skip the local card check, and the launcher reads it to spawn the function.
os.environ["NANOCHAT_TARGET"] = "modal"
# The run directory keeps its name on the Volume. Without this line the local
# path /kaggle/working/runs/base would become /runs/runs/base.
os.environ["NANOCHAT_RUNS_DIR"] = str(RUNS)

missing = [name for name in ("MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET")
           if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"add the Kaggle secrets {', '.join(missing)}")

import modal
import target

# The preflight checks the two account-level prerequisites. It does not create
# them: a deployment belongs to the commit, not to the session, and an image
# rebuild per session is wasted container time.
try:
    modal.Function.from_name(target.APP_NAME, target.FUNCTION)
    print("modal app:", target.APP_NAME, "is deployed")
except Exception as error:
    raise RuntimeError(
        "deploy the app before this notebook, once per commit: "
        "modal deploy tools/modal/app.py") from error

try:
    modal.Secret.from_name(target.SECRET_NAME)
    print("modal secret:", target.SECRET_NAME, "is present")
except Exception as error:
    raise RuntimeError(
        "the publish needs the Hub token, once per account: "
        "modal secret create huggingface HF_TOKEN=hf_... "
        "(or drop publish= from the launch call to skip the publish)"
    ) from error

sess = session.bootstrap(REPO_URL, REPO, arch="sm_75", devices=1, base=BASE)
session.describe(sess)

import hub_checkpoint
import nanochat_cpp as nc

## Data and tokenizer

Stage one ClimbMix train shard and one validation shard, then pull the
tokenizer from the Hub. Both live under the run root, `RUNS`, because the
launcher uploads everything the job reads from that one root. On the Volume the
same names appear under `/runs`: `/runs/data` for the shards and
`/runs/tokenizer` for the tokenizer.

A pull miss trains the tokenizer and pushes it. The download is the one fixed
cost of the session.

In [ ]:
import shutil

VOCAB_SIZE = 32768
DATA = RUNS / "data"
TRAIN_SHARD = hub_checkpoint.TRAIN_SHARD
VAL_SHARD = hub_checkpoint.VAL_SHARD

if not (DATA / VAL_SHARD).is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "huggingface_hub"], check=True)
hub_checkpoint.download_climbmix(DATA, token=os.environ.get("HF_TOKEN"))

TOKENIZER = RUNS / "tokenizer" / "tokenizer.nctoken"
TOKENIZER.parent.mkdir(parents=True, exist_ok=True)
TOKENIZER_REPO = hub_checkpoint.repo_id_for("tokenizer")

if not TOKENIZER.is_file():
    pulled = hub_checkpoint.pull_tokenizer(
        TOKENIZER_REPO, VOCAB_SIZE, TOKENIZER.parent,
        token=os.environ.get("HF_TOKEN"))
    if pulled is not None:
        shutil.copyfile(pulled, TOKENIZER)
        print("pulled tokenizer:", pulled)
    else:
        print("no tokenizer on the Hub; training one")
        nc.toolchain.build(["//src/tokenizer:tok_train_main"])
        nc.toolchain.run("t0-cpu", [
            "bazel-bin/src/tokenizer/tok_train_main",
            "--parquet", f"{DATA}/{TRAIN_SHARD}",
            "--vocab-size", str(VOCAB_SIZE),
            "--max-chars", "2000000000",
            "--out", str(TOKENIZER),
        ])
        hub_checkpoint.push_tokenizer(TOKENIZER_REPO, VOCAB_SIZE, TOKENIZER,
                                      token=os.environ.get("HF_TOKEN"))

tokenizer = nc.Tokenizer.load(str(TOKENIZER))
assert tokenizer.decode(tokenizer.encode("The capital of France is")) == \
    "The capital of France is", "the tokenizer round trip failed"
print("train:", DATA / TRAIN_SHARD)
print("tokenizer vocab:", tokenizer.vocab_size)

## Model

The default is the balanced shape from `pretrain.ipynb`: 16 layers, hidden
1024, 8 heads, 4 key/value heads, and 4096 tokens, about 252M parameters. Keep
`num_heads` and `num_kv_heads` as divisors of `hidden_dim`.

In [ ]:
import args
import model_fit

MODEL = args.ModelArgs(num_layers=16, hidden_dim=1024, num_heads=8,
                       num_kv_heads=4, seq_len=4096, vocab_size=VOCAB_SIZE)
spec = model_fit.spec_from(MODEL)
config = MODEL.to_config()

counts = model_fit.parameter_counts(spec)
t4 = model_fit.hardware("t4")
print("parameters:", f"{counts['total']:,}")
print("flops per token:", f"{model_fit.flops_per_token(spec) / 1e9:.3f} GFLOP")
print("estimated card memory at per-rank batch 1:",
      f"{model_fit.card_bytes(spec, 1, 'fp32') / 2**30:.2f} GiB of 16 GiB")

## Train on Modal

The image holds the code, so this cell uploads only the data and the tokenizer.
The job then runs on one T4 card in a Modal container.

`world_size` must equal `GPU_COUNT` in `tools/modal/app.py`. The app checks the
replicas against the cards before the run, so a mismatch stops early. The
default is one card, because the deployed image is built for `T4` (`sm_75`).
`grad_accum` is 16, so the global sequence count per step matches the two-card
run: 1 x 16 x 1 = 16. The loss curve stays comparable, and each step takes
about twice as long.

A larger card set is two edits and one deploy. Set `GPU_TYPE`, `GPU_COUNT`, and
`CUDA_ARCH` in `tools/modal/app.py`, then set `world_size` here to the same
count and lower `grad_accum` to keep the product at 16.

The publish runs in the same container, after the training. The `publish`
dictionary carries only what the notebook alone knows: the model arguments, the
job, the repository, the tag, and the Git revision. The container adds the build
key and the architecture, because the container holds the library that trained
the model.

The cell streams the worker log. The container commits the Volume every 10
seconds, so the log and the metrics arrive during the run. Interrupt the cell to
cancel the run; the last committed checkpoint is the resume point.

The Kaggle cards stay idle during the run. The session uses them only for the
local library build.

In [ ]:
import json

import mesh_train

BASE_STEPS = 500
RUN_NAME = (f"base-L{MODEL.num_layers}-H{MODEL.hidden_dim}"
            f"-n{MODEL.num_heads}-kv{MODEL.num_kv_heads}"
            f"-s{MODEL.seq_len}")
REPO_ID = hub_checkpoint.repo_id_for(RUN_NAME)

train_config = nc.MeshTrainConfig(
    MODEL.to_config(), preset="track3", world_size=1,
    tokenizer=str(TOKENIZER), train_parquet=f"{DATA}/{TRAIN_SHARD}",
    per_rank_batch=1, grad_accum=16, steps=BASE_STEPS,
    seed=42, data_seed=42, log_every=1, save_every=50,
    resume=True, out_dir=str(RUNS / RUN_NAME),
    flops_per_token=model_fit.flops_per_token(spec),
    peak_flops=t4.peak_flops("fp32"))
print("run:", train_config.out_dir, "| tokens per step:",
      train_config.tokens_per_step)
print("remote run directory:", f"/runs/{RUN_NAME}")

# The stage folder and the upload happen beside the checkpoint, in the
# container. The checkpoint never crosses this notebook.
publish = {
    "stage": "base",
    "run_name": RUN_NAME,
    "repo_id": REPO_ID,
    "tag": f"base-{BASE_STEPS}",
    "revision": sess.revision,
    "step": BASE_STEPS,
    "description": f"Base model, pretrained on ClimbMix for {BASE_STEPS} steps.",
    "model_json": MODEL.to_json(),
    "train_json": train_config.job_json(),
}

live = mesh_train.launch(train_config, root=REPO, publish=publish)
live.stream()

print("worker exit code:", live.wait())

# The per-step log streamed above. This block prints the run summary.
run_result = json.loads(live.result())
summary = run_result["summary"]
records = live.rows()
print(f"steps {len(records)}  "
      f"mean loss {float(summary.get('mean_loss', 0.0)):.4f}  "
      f"steady MFU {float(summary.get('steady_mfu', 0.0)) * 100:.2f}%")
if summary:
    digests = summary["replica_digests"]
    print("replica digests:", digests)
    assert len(digests) == 1, "one replica expected on one card"

## Publish

The container wrote the stage folder beside the checkpoint, then uploaded it to
`<repository>/base` and tagged it. The token comes from the Modal Secret
`huggingface`, which holds `HF_TOKEN`.

No bytes cross this notebook. The cell prints the record the container wrote
into `run.json`.

In [ ]:
published = run_result["publish"]
print("published:", published["repo_id"], "stage", published["stage"],
      "tag", published["tag"])
print(published["record"])

## Retrieve

The checkpoint stays on the runs Volume. Pull the run directory only when you
want to inspect the files, or to load the checkpoint in this session. A CPU
session skips this cell and the sanity check.

In [ ]:
run_dir = Path(train_config.out_dir)
written = live.pull(run_dir)
print("pulled:", sorted(path.name for path in written))
checkpoint = train_config.checkpoint_path()
print("checkpoint:", checkpoint, checkpoint.stat().st_size, "bytes")

## Sanity check

This cell needs a local card. It loads the pulled checkpoint into one
`nc.Model`, measures a few validation steps, and prints two greedy samples. In a
CPU session, run `eval.ipynb` in a later GPU session instead.

In [ ]:
import kaggle_demo

assert nc.devices(), "this cell needs a local card; run eval.ipynb instead"

model = nc.Model(config, device="cuda", seed=0)
model.load(str(checkpoint))
print("device:", model.device, "parameters:", f"{model.param_total():,}")

report = nc.evaluate(model, parquet=str(DATA / VAL_SHARD), tokenizer=tokenizer,
                     tokens=config.seq_len * 4, batch=1, seed=43)
print(f"validation bits per byte: {report.bpb:.4f}")

kaggle_demo.print_samples(model, tokenizer,
                          kaggle_demo.SAMPLE_PROMPTS[:2])

## Next

Run `eval.ipynb` in a later session. It downloads the tag `base-500` and
measures the base stage. Run `posttrain.ipynb` to fine-tune it into the `sft`
stage.

The run directory also stays on the Modal Volume `nanochat-runs` under
`/runs/<run name>`. Pull it in a later session with:

```bash
modal volume get nanochat-runs <run name> ./<run name>
```